# **1. Setup and Initialization**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import re
import json
import random
import numpy as np
import torch

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [3]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

# **2. Loading the Baseline**

In [4]:
from transformers import MarianMTModel, MarianTokenizer

model     = MarianMTModel.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")
tokeniser = MarianTokenizer.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")
model     = model.to(device)

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [5]:
def translate_batch(texts, target_lang=">>mal<<", batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch  = [f"{target_lang} {t}" for t in texts[i:i+batch_size]]
        inputs = tokeniser(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            generated = model.generate(**inputs, num_beams=4, max_length=256)
        decoded = tokeniser.batch_decode(generated, skip_special_tokens=True)
        results.extend(decoded)
    return results

# Sanity check
print(translate_batch(["Where is the nearest hospital?"])[0])
print(translate_batch(["We propose a novel attention mechanism for neural machine translation."])[0])

ഏറ്റവും അടുത്തുള്ള ആശുപത്രി എവിടെയാണ്?
ന്യൂറൽ മെഷീൻ വിവർത്തനത്തിനായുള്ള നോവൽ ശ്രദ്ധാ സംവിധാനം ഞങ്ങൾ നിർദ്ദേശിക്കുന്നു.


# **3. Technical Dataset: Shiksha Dataset**

In [6]:
from datasets import load_dataset

shiksha = load_dataset(
    "SPRINGLab/shiksha",
    trust_remote_code=True,
    streaming=False
)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'SPRINGLab/shiksha' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'SPRINGLab/shiksha' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/195M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/195M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/85.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2519061 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/277715 [00:00<?, ? examples/s]

In [7]:
print(shiksha)
print(next(iter(shiksha['train'])))

DatasetDict({
    train: Dataset({
        features: ['src_lang', 'tgt_lang', 'src_text', 'tgt_text', 'score', 'course_id', 'lecture_id'],
        num_rows: 2519061
    })
    test: Dataset({
        features: ['src_lang', 'tgt_lang', 'src_text', 'tgt_text', 'score', 'course_id', 'lecture_id'],
        num_rows: 277715
    })
})
{'src_lang': 2, 'tgt_lang': 7, 'src_text': 'લગભગ અડધી, 3 થી 3.5 કિ. મી. પ્રતિ સેકન્ડ.', 'tgt_text': 'வேகம் மெதுவாக, கிட்டத்தட்ட பாதி, வினாடிக்கு 3 முதல் 3.5 கி.', 'score': 0.8708644, 'course_id': '105104152', 'lecture_id': '13'}


In [8]:
import pandas as pd
df = shiksha['train'].to_pandas()
print(df['src_lang'].unique())
print(df['tgt_lang'].unique())

[2 0 3 6 1 5 7 4]
[7 3 6 8 5 2 4 1]


In [9]:
mal_data = shiksha['train'].filter(
    lambda x: x['src_lang'] == 0 and x['tgt_lang'] == 5
)
print(f"English→Malayalam pairs: {len(mal_data)}")
print(mal_data[0])

Filter:   0%|          | 0/2519061 [00:00<?, ? examples/s]

English→Malayalam pairs: 135080
{'src_lang': 0, 'tgt_lang': 5, 'src_text': 'So, what you can write now?', 'tgt_text': 'അതിനാൽ, നിങ്ങൾക്ക് ഇപ്പോൾ എന്താണ് എഴുതാൻ കഴിയുക?', 'score': 0.9331815, 'course_id': '108104139', 'lecture_id': '54'}


In [10]:
# Quality filter first
mal_data = shiksha['train'].filter(
    lambda x: x['src_lang'] == 0 and x['tgt_lang'] == 5 and x['score'] > 0.85
)
print(f"After quality filter: {len(mal_data)}")


Filter:   0%|          | 0/2519061 [00:00<?, ? examples/s]

After quality filter: 54395


In [11]:
shiksha_split = mal_data.train_test_split(test_size=0.2, seed=42)
shiksha_train = shiksha_split["train"]
shiksha_val   = shiksha_split["test"]

print(f"Shiksha train: {len(shiksha_train):,}  |  val: {len(shiksha_val):,}")

Shiksha train: 43,516  |  val: 10,879


# **4. Preprocessing and Tokenization**

In [12]:
MAX_LEN = 256

def preprocess_batch(batch):
    src = [f">>mal<< {s}" for s in batch["src_text"]]
    tgt = batch["tgt_text"]

    model_inputs = tokeniser(
        src,
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

    labels = tokeniser(
        text_target=tgt,
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenised_shiksha = {}
for name, data in [("train", shiksha_train), ("validation", shiksha_val)]:
    tokenised_shiksha[name] = data.map(
        preprocess_batch,
        batched=True,
        remove_columns=data.column_names,
    )
    print(f"{name}: {len(tokenised_shiksha[name]):,} tokenised")

Map:   0%|          | 0/43516 [00:00<?, ? examples/s]

train: 43,516 tokenised


Map:   0%|          | 0/10879 [00:00<?, ? examples/s]

validation: 10,879 tokenised


In [13]:
# Sanity check
print(tokenised_shiksha["train"][0])

{'input_ids': [13, 252, 3, 35682, 3281, 28360, 384, 6957, 2921, 23, 75, 5, 4, 2921, 2933, 59, 6044, 55274, 516, 4, 2827, 5, 85, 37737, 7082, 2, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [36276, 3, 117, 28725, 831, 14017, 19130, 17, 17691, 1215, 3198, 1932, 18337, 898, 5627, 655, 7234, 503, 3611, 377, 143, 18137, 437, 17, 7304, 12636, 1421, 17223, 3727, 499, 28725, 831, 23369, 7304, 90, 3161, 831, 17, 7304, 12636, 408, 2, 0]}


# **5. Further Fine-Tuning**

In [14]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.4 MB/s eta 0:00:00


In [15]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

import sacrebleu

data_collator = DataCollatorForSeq2Seq(
    tokeniser,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

from sacrebleu.metrics import BLEU
bleu_metric = BLEU()

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokeniser.pad_token_id)
    decoded_preds  = tokeniser.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokeniser.batch_decode(labels, skip_special_tokens=True)
    result = bleu_metric.corpus_score(decoded_preds, [decoded_labels])
    return {"bleu": round(result.score, 2)}


training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/marian_shiksha",
    num_train_epochs=4,
    per_device_train_batch_size=32,   # Marian is small, can handle larger batches
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=256,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenised_shiksha["train"],
    eval_dataset=tokenised_shiksha["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,1.476989,1.256926,28.930000
2,1.246602,1.153274,31.180000
3,1.145933,1.117021,32.240000
4,1.096816,1.105637,32.650000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=5440, training_loss=1.304932128681856, metrics={'train_runtime': 4634.1846, 'train_samples_per_second': 37.561, 'train_steps_per_second': 1.174, 'total_flos': 4495440139517952.0, 'train_loss': 1.304932128681856, 'epoch': 4.0})

In [16]:
model.save_pretrained("/content/drive/MyDrive/marian_shiksha_final")
tokeniser.save_pretrained("/content/drive/MyDrive/marian_shiksha_final")
print("Saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


In [17]:
print(os.listdir("/content/drive/MyDrive/marian_shiksha_final"))

['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']


# **6. Evaluating the Model**

In [4]:
from transformers import MarianMTModel, MarianTokenizer

model     = MarianMTModel.from_pretrained("/content/drive/MyDrive/marian_shiksha_final")
tokeniser = MarianTokenizer.from_pretrained("/content/drive/MyDrive/marian_shiksha_final")
model     = model.to(device)

def translate_batch(texts, target_lang=">>mal<<", batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch  = [f"{target_lang} {t}" for t in texts[i:i+batch_size]]
        inputs = tokeniser(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            generated = model.generate(**inputs, num_beams=4, max_length=256)
        decoded = tokeniser.batch_decode(generated, skip_special_tokens=True)
        results.extend(decoded)
    return results

# Sanity check
print(translate_batch(["The cat sat on the mat."])[0])
print(translate_batch(["We propose a novel attention mechanism for neural machine translation."])[0])

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


പൂച്ച മോഡിൽ ഇരുന്നു.
ന്യൂറൽ മെഷീൻ വിവർത്തനത്തിനായുള്ള നോവൽ ശ്രദ്ധാ കേന്ദ്ര സംവിധാനം ഞങ്ങൾ നിർദ്ദേശിക്കുന്നു.


In [25]:
from sacrebleu.metrics import BLEU

bleu = BLEU()

sources    = shiksha_val['src_text']
references = shiksha_val['tgt_text']

hypotheses = translate_batch(sources)
score      = bleu.corpus_score(hypotheses, [references])
print(f"Baseline BLEU: {score}")

Baseline BLEU: BLEU = 32.61 66.6/40.9/26.2/17.4 (BP = 0.976 ratio = 0.976 hyp_len = 218142 ref_len = 223401)


In [20]:
import re

def clean_abstract(text):
    text = re.sub(r'\$\$.*?\$\$', '', text)
    text = re.sub(r'\$.*?\$', '', text)
    text = re.sub(r'\\[a-zA-Z]+\{.*?\}', '', text)
    text = re.sub(r'\\[a-zA-Z]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

from datasets import load_dataset

arxiv  = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
sample = arxiv.shuffle(seed=SEED).select(range(500))

# Combine title + abstract, clean LaTeX
clean_texts = []
for row in sample:
    title    = clean_abstract(row['title'])
    abstract = clean_abstract(row['abstract'])
    combined = f"{title}. {abstract}".strip()
    if len(combined) > 50:   # drop entries that are mostly math
        clean_texts.append(combined)

print(f"Clean abstracts: {len(clean_texts)}")
print(clean_texts[0][:300])

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

Clean abstracts: 500
Epsilon Consistent Mixup: Structural Regularization with an Adaptive Consistency-Interpolation Tradeoff. In this paper we propose -Consistent Mixup (mu). mu is a data-based structural regularization technique that combines Mixup's linear interpolation with consistency regularization in the Mixup dir


In [22]:
arxiv_translations = translate_batch(clean_texts)

import json
results = [{"src": en, "tgt": ml}
           for en, ml in zip(clean_texts, arxiv_translations)]

with open("/content/drive/MyDrive/arxiv_shiksha_translations.json", "w") as f:
    json.dump(results, f, ensure_ascii=False)
print(f"Saved {len(results)} translations")

Saved 500 translations


In [24]:
with open("/content/drive/MyDrive/arxiv_shiksha_translations.json") as f:
    data = json.load(f)

print(f"Total pairs: {len(data)}")
print("\nFirst pair:")
print("EN:", data[0]['src'][:200])
print("ML:", data[0]['tgt'][:200])

print("\nTechnical pair example:")
technical = [x for x in data if any(t in x['src'].lower()
             for t in ["attention", "gradient", "embedding", "transformer"])]
print("EN:", technical[0]['src'][:200])
print("ML:", technical[0]['tgt'][:200])

Total pairs: 500

First pair:
EN: Epsilon Consistent Mixup: Structural Regularization with an Adaptive Consistency-Interpolation Tradeoff. In this paper we propose -Consistent Mixup (mu). mu is a data-based structural regularization t
ML: എപിസിലോൺ മിക്സഡ് മിക്സഡ് മിക്സേഷൻ: ഒരു അഡാപ്റ്റീവ് സിനിമേറ്റഡ്-II ഇന്റർപോണൻസിറ്റി ട്രേഡുള്ള സ്ട്രക്ച്ചറൽ റെസിസ്റ്റൻഷ്യലൈസേഷൻ. ഈ പേപ്പറിൽ നമ്മൾ നിർദ്ദേശിക്കുന്ന ഡാറ്റാ അധിഷ്ഠിത ഘടനാപരമായ ഘടനാ സംവിധാന സ

Technical pair example:
EN: A novel multi-scale loss function for classification problems in machine learning. We introduce two-scale loss functions for use in various gradient descent algorithms applied to classification proble
ML: മെഷീൻ ലേണിംഗിൽ വർഗ്ഗീകരണ പ്രശ്നങ്ങൾക്കുള്ള ഒരു നോവൽ-സ് ട്രെയിൻ ലോസ് ഫംഗ്ഷൻ. വിവിധ ഗ്രേഡിയന്റ് പാരമ്പര്യ അൽഗോരിത പ്രവർത്തനങ്ങളിൽ ഉപയോഗിക്കുന്നതിന് ഞങ്ങൾ രണ്ട്-മീറ്റർ ലോഹ പ്രവർത്തനങ്ങൾ അവതരിപ്പിക്കുന്നു
